# Kaggle Submission Notebook

This notebook keeps the same retrieval logic, but reorganizes the code into clear classes for runtime resolution, data loading, preprocessing, retrieval, and submission writing.


In [ ]:
import importlib.util

REQUIRED_LIBRARIES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'rank_bm25': 'rank-bm25',
    'sklearn': 'scikit-learn',
    'sentence_transformers': 'sentence-transformers',
    'torch': 'torch',
    'nltk': 'nltk',
}

missing_packages = []
for module_name, package_name in REQUIRED_LIBRARIES.items():
    installed = importlib.util.find_spec(module_name) is not None
    status = 'OK' if installed else 'MISSING'
    print(f'[{status}] {package_name}')
    if not installed:
        missing_packages.append(package_name)

if missing_packages:
    install_cmd = 'pip install ' + ' '.join(missing_packages)
    raise ImportError(
        'Missing required libraries. Install them before running the notebook:\n'
        f'{install_cmd}'
    )

print('All required libraries are installed.')


In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Sequence
import csv
import json
import logging
import re
import sys
import time
import unicodedata

import numpy as np
import pandas as pd
from rank_bm25 import BM25Plus
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class StructuredLogger:
    def __init__(self, name: str = 'kaggle_submission') -> None:
        self._logger = logging.getLogger(name)
        self._logger.setLevel(logging.INFO)
        self._logger.handlers.clear()
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter('[%(levelname)s] %(asctime)s | %(message)s', '%H:%M:%S'))
        self._logger.addHandler(handler)
        self._logger.propagate = False

    def info(self, message: str) -> None:
        self._logger.info(message)

    def warning(self, message: str) -> None:
        self._logger.warning(message)

    @contextmanager
    def section(self, title: str):
        started_at = time.perf_counter()
        self.info(f'{title} | started')
        try:
            yield
        finally:
            elapsed = time.perf_counter() - started_at
            self.info(f'{title} | finished in {elapsed:.2f}s')


def format_bytes(num_bytes: int | float) -> str:
    value = float(num_bytes)
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024
    return f'{value:.2f} TB'


@dataclass(slots=True)
class SparseConfig:
    lowercase: bool = True
    strip_noise: bool = True
    collapse_whitespace: bool = True
    remove_stopwords: bool = False
    normalizer: str = 'stem'
    normalize_numbers: bool = True
    normalize_units: bool = True
    normalize_hyphens: bool = True


@dataclass(slots=True)
class EmbeddingConfig:
    model_name: str = 'sentence-transformers/all-MiniLM-L6-v2'
    batch_size: int = 128
    candidate_k: int = 200
    device: str = 'auto'


@dataclass(slots=True)
class PipelineConfig:
    project_folder_name: str = 'retrieval_project'
    output_path: Path = Path('notebooks/kaggle/solutions_SeaFour.csv')
    model_name: str = 'embedding_hybrid'
    top_k: int = 100
    field_weights: Dict[str, float] = field(default_factory=lambda: {'title': 3.0, 'text': 1.0, 'tags': 2.0})
    sparse: SparseConfig = field(default_factory=SparseConfig)
    embedding: EmbeddingConfig = field(default_factory=EmbeddingConfig)


class RuntimeEnvironment:
    def __init__(self, config: PipelineConfig, logger: StructuredLogger) -> None:
        self.config = config
        self.logger = logger

    @staticmethod
    def is_colab() -> bool:
        try:
            import google.colab  # type: ignore
            return True
        except ImportError:
            return False

    def maybe_mount_google_drive(self) -> None:
        try:
            import google.colab  # type: ignore
            from google.colab import drive  # type: ignore
        except ImportError:
            return

        if not Path('/content/drive/MyDrive').exists():
            self.logger.info('Google Colab detected. Mounting Google Drive.')
            drive.mount('/content/drive')
        else:
            self.logger.info('Google Drive already mounted.')

    def candidate_data_dirs(self) -> List[Path]:
        cwd = Path.cwd().resolve()
        candidates = [
            Path('/kaggle/input/retrieval-engine-competition'),
            cwd / 'data',
            cwd.parent / 'data',
            cwd.parent.parent / 'data',
        ]

        for parent in (cwd, *cwd.parents):
            if parent.name == self.config.project_folder_name:
                candidates.append(parent / 'data')

        drive_root = Path('/content/drive/MyDrive')
        if drive_root.exists():
            candidates.append(drive_root / self.config.project_folder_name / 'data')
            candidates.append(drive_root / 'Colab Notebooks' / self.config.project_folder_name / 'data')
            candidates.extend(path / 'data' for path in drive_root.glob(f'*/{self.config.project_folder_name}'))

        unique_candidates: List[Path] = []
        seen = set()
        for candidate in candidates:
            candidate = candidate.expanduser()
            if candidate in seen:
                continue
            seen.add(candidate)
            unique_candidates.append(candidate)
        return unique_candidates

    def resolve_data_dir(self) -> Path:
        self.maybe_mount_google_drive()
        self.logger.info(f'Current working directory: {Path.cwd().resolve()}')
        self.logger.info(f'Running in Colab: {self.is_colab()}')
        self.logger.info('Searching for dataset directory.')

        for candidate in self.candidate_data_dirs():
            self.logger.info(f'Checking data directory candidate: {candidate}')
            if (candidate / 'docs.json').exists() and (candidate / 'queries_test.json').exists():
                self.logger.info(f'Using dataset directory: {candidate}')
                return candidate

        searched = '\n'.join(str(path) for path in self.candidate_data_dirs())
        raise FileNotFoundError(
            'Could not find the dataset directory. Checked:\n'
            f'{searched}\n\n'
            'Expected docs.json and queries_test.json under the data/ directory.'
        )

    def resolve_project_root(self, data_dir: Path) -> Path:
        for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
            if parent.name == self.config.project_folder_name:
                return parent
        for parent in (data_dir.resolve(), *data_dir.resolve().parents):
            if parent.name == self.config.project_folder_name:
                return parent
        return Path.cwd().resolve()

    def resolve_output_path(self, data_dir: Path) -> Path:
        output_path = self.config.output_path
        if output_path.is_absolute():
            return output_path
        return self.resolve_project_root(data_dir) / output_path

    def resolve_model_name(self) -> str:
        if self.config.model_name == 'auto':
            return 'embedding_hybrid'
        return self.config.model_name

    def resolve_embedding_device(self) -> str:
        preferred = self.config.embedding.device
        if preferred != 'auto':
            return preferred

        import torch

        if torch.cuda.is_available():
            return 'cuda'
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return 'mps'
        return 'cpu'


In [ ]:
@dataclass(slots=True)
class SubmissionDataset:
    docs_df: pd.DataFrame
    queries_df: pd.DataFrame
    sample_submission_path: Path


class JsonDatasetLoader:
    def __init__(self, logger: StructuredLogger) -> None:
        self.logger = logger

    def load_dataframe(self, path: Path) -> pd.DataFrame:
        file_size = path.stat().st_size if path.exists() else 0
        self.logger.info(f'Loading JSON file: {path} ({format_bytes(file_size)})')

        pandas_error = None
        try:
            df = pd.read_json(path)
            self.logger.info(f'Loaded {len(df):,} rows x {len(df.columns)} columns from {path.name} via pandas.read_json')
            return df
        except ValueError as error:
            pandas_error = str(error)
            self.logger.warning(f'pandas.read_json failed for {path.name}. Falling back to json.load.')

        try:
            with path.open('r', encoding='utf-8') as handle:
                payload = json.load(handle)
        except json.JSONDecodeError as error:
            preview = path.read_text(encoding='utf-8', errors='replace')[:500]
            raise ValueError(
                f'Failed to parse JSON file: {path}\n'
                f'file size: {file_size} bytes\n'
                f'pandas error: {pandas_error}\n'
                f'json error: {error}\n'
                f'File preview:\n{preview}'
            ) from error

        if isinstance(payload, list):
            df = pd.DataFrame(payload)
            self.logger.info(f'Loaded {len(df):,} rows x {len(df.columns)} columns from {path.name} via json.load(list)')
            return df
        if isinstance(payload, dict):
            df = pd.DataFrame.from_dict(payload, orient='index').reset_index(names='id')
            self.logger.info(f'Loaded {len(df):,} rows x {len(df.columns)} columns from {path.name} via json.load(dict)')
            return df

        raise ValueError(f'Unsupported JSON top-level type in {path}: {type(payload).__name__}')

    def load_submission_dataset(self, data_dir: Path) -> SubmissionDataset:
        with self.logger.section('Dataset loading'):
            docs_df = self.load_dataframe(data_dir / 'docs.json')
            queries_df = self.load_dataframe(data_dir / 'queries_test.json')
            sample_submission_path = data_dir / 'submission.csv'
            self.logger.info(f'Sample submission path: {sample_submission_path}')
            return SubmissionDataset(docs_df=docs_df, queries_df=queries_df, sample_submission_path=sample_submission_path)


class TextPreprocessor:
    SAFE_STOPWORDS = set(ENGLISH_STOP_WORDS) - {'no', 'nor', 'not'}
    TOKEN_PATTERN = re.compile(r'[a-z0-9]+(?:_[a-z0-9]+)*')

    def __init__(self, config: SparseConfig, logger: StructuredLogger) -> None:
        self.config = config
        self.logger = logger
        self._stemmer = None
        self._lemmatizer = None

    @staticmethod
    def value_to_text(value: object) -> str:
        if value is None:
            return ''
        if isinstance(value, (list, tuple)):
            return ' '.join(str(item) for item in value)
        if pd.isna(value):
            return ''
        return str(value)

    def create_content_column(self, df: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
        self.logger.info(f'Creating merged content column from fields: {list(columns)}')
        out = df.copy()
        for column in columns:
            if column not in out.columns:
                out[column] = ''

        merged = [
            ' '.join(self.value_to_text(value) for value in row).strip()
            for row in out[list(columns)].itertuples(index=False, name=None)
        ]
        out['content'] = merged
        out['id'] = out['id'].astype(str)
        self.logger.info(f'Created content column for {len(out):,} rows')
        return out

    def _get_stemmer(self):
        if self._stemmer is None:
            from nltk.stem import SnowballStemmer
            self._stemmer = SnowballStemmer('english')
        return self._stemmer

    def _get_lemmatizer(self):
        if self._lemmatizer is None:
            import nltk
            from nltk.stem import WordNetLemmatizer

            try:
                nltk.data.find('corpora/wordnet')
            except LookupError:
                self.logger.info('Downloading NLTK WordNet resources.')
                nltk.download('wordnet', quiet=True)
                nltk.download('omw-1.4', quiet=True)

            self._lemmatizer = WordNetLemmatizer()
        return self._lemmatizer

    def normalize_sparse_text(self, text: object) -> str:
        cfg = self.config
        txt = unicodedata.normalize('NFKC', str(text or ''))

        if cfg.lowercase:
            txt = txt.lower()

        txt = re.sub(r'\be[\s\-_]*mail\b', ' email ', txt)
        txt = re.sub(r'\bcovid[\s\-_]*19\b', ' covid19 covid 19 ', txt)

        if cfg.normalize_hyphens:
            txt = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2212]', '-', txt)
            txt = re.sub(r'\b([a-z]+)[\-_](\d+[a-z0-9]*)\b', r' \1\2 \1 \2 ', txt)
            txt = re.sub(r'\b(\d+[a-z0-9]*)[\-_]([a-z]+)\b', r' \1\2 \1 \2 ', txt)
            txt = re.sub(r'(?<=[a-z])[-_/](?=[a-z])', ' ', txt)

        if cfg.normalize_numbers:
            txt = re.sub(r'(?<=\d),(?=\d)', '', txt)
            txt = re.sub(r'(?<=\d)[./](?=\d)', '_', txt)
            txt = re.sub(r'\b(v(?:ersion)?)\s*(\d+(?:[_]\d+)*)\b', r' \1\2 \1 \2 ', txt)

        if cfg.normalize_units:
            txt = re.sub(r'\b(\d+(?:[_]\d+)?)\s*([a-z]{1,5})\b', r' \1\2 \1 \2 ', txt)

        if cfg.strip_noise:
            txt = re.sub(r'[^a-z0-9_\s.]', ' ', txt)

        if cfg.collapse_whitespace:
            txt = re.sub(r'\s+', ' ', txt).strip()

        return txt

    def tokenize(self, text: object) -> List[str]:
        tokens = self.TOKEN_PATTERN.findall(self.normalize_sparse_text(text))

        if self.config.remove_stopwords:
            tokens = [token for token in tokens if token not in self.SAFE_STOPWORDS]

        if self.config.normalizer == 'stem':
            stemmer = self._get_stemmer()
            tokens = [stemmer.stem(token) for token in tokens]
        elif self.config.normalizer == 'lemma':
            lemmatizer = self._get_lemmatizer()
            tokens = [lemmatizer.lemmatize(token) for token in tokens]
        elif self.config.normalizer != 'raw':
            raise ValueError("SparseConfig.normalizer must be 'raw', 'stem', or 'lemma'.")

        return tokens

    def build_sparse_content_column(self, df: pd.DataFrame, content_col: str = 'content', output_col: str = 'sparse_content') -> pd.DataFrame:
        with self.logger.section(f'Build sparse column: {output_col}'):
            out = df.copy()
            out[output_col] = out[content_col].map(lambda value: ' '.join(self.tokenize(self.value_to_text(value))))
            return out

    def build_sparse_field_columns(self, df: pd.DataFrame, fields: Sequence[str]) -> pd.DataFrame:
        with self.logger.section('Build sparse field columns'):
            out = df.copy()
            for field in fields:
                if field not in out.columns:
                    out[field] = ''
                sparse_col = f'sparse_{field}'
                out[sparse_col] = out[field].map(lambda value: ' '.join(self.tokenize(self.value_to_text(value))))
                self.logger.info(f'Built sparse column: {sparse_col}')
            return out


In [ ]:
def top_k_indices_desc(scores: np.ndarray, top_k: int) -> np.ndarray:
    top_k = min(top_k, len(scores))
    if top_k <= 0:
        return np.array([], dtype=int)
    if top_k == len(scores):
        return np.argsort(scores)[::-1]
    indices = np.argpartition(scores, -top_k)[-top_k:]
    return indices[np.argsort(scores[indices])[::-1]]


def build_result(query_id: str, ranked_doc_ids: Sequence[str]) -> dict:
    return {'query_id': str(query_id), 'relevant_docs': [str(doc_id) for doc_id in ranked_doc_ids]}


class WeightedTfidfRetriever:
    def __init__(self, field_weights: Dict[str, float], logger: StructuredLogger) -> None:
        self.field_weights = field_weights
        self.logger = logger

    @staticmethod
    def _make_vectorizer(min_df: int) -> TfidfVectorizer:
        return TfidfVectorizer(
            lowercase=False,
            preprocessor=None,
            tokenizer=str.split,
            token_pattern=None,
            ngram_range=(1, 2),
            min_df=min_df,
            dtype=np.float32,
        )

    def _compute_scores(self, doc_texts: pd.Series, query_texts: pd.Series) -> np.ndarray:
        vectorizer = self._make_vectorizer(min_df=2)
        try:
            doc_vectors = vectorizer.fit_transform(doc_texts)
        except ValueError as error:
            if 'After pruning, no terms remain' not in str(error):
                raise
            vectorizer = self._make_vectorizer(min_df=1)
            doc_vectors = vectorizer.fit_transform(doc_texts)
        query_vectors = vectorizer.transform(query_texts)
        return cosine_similarity(query_vectors, doc_vectors).astype(np.float32, copy=False)

    def search(self, docs_df: pd.DataFrame, queries_df: pd.DataFrame, top_k: int, query_col: str = 'sparse_content') -> List[dict]:
        with self.logger.section('TF-IDF retrieval'):
            self.logger.info(f'Queries: {len(queries_df):,} | Docs: {len(docs_df):,} | top_k: {top_k}')
            query_texts = queries_df[query_col].fillna('')
            scores = np.zeros((len(queries_df), len(docs_df)), dtype=np.float32)
            doc_ids = docs_df['id'].astype(str).to_numpy()

            for field, weight in self.field_weights.items():
                if weight <= 0:
                    continue
                doc_col = f'sparse_{field}'
                if doc_col not in docs_df.columns:
                    continue
                doc_texts = docs_df[doc_col].fillna('')
                if doc_texts.str.len().eq(0).all():
                    continue
                self.logger.info(f'TF-IDF field: {field} | weight={weight}')
                scores += weight * self._compute_scores(doc_texts, query_texts)

            results = []
            for row_idx, row_scores in enumerate(scores):
                top_idx = top_k_indices_desc(row_scores, top_k)
                results.append(build_result(queries_df.iloc[row_idx]['id'], doc_ids[top_idx].tolist()))
            return results


class WeightedBM25Retriever:
    def __init__(self, field_weights: Dict[str, float], logger: StructuredLogger, k1: float = 1.5, b: float = 0.75, delta: float = 1.0) -> None:
        self.field_weights = field_weights
        self.logger = logger
        self.k1 = k1
        self.b = b
        self.delta = delta

    def search(self, docs_df: pd.DataFrame, queries_df: pd.DataFrame, top_k: int, query_col: str = 'sparse_content') -> List[dict]:
        with self.logger.section('BM25 retrieval'):
            self.logger.info(f'Queries: {len(queries_df):,} | Docs: {len(docs_df):,} | top_k: {top_k}')
            field_models = []
            doc_ids = docs_df['id'].astype(str).to_numpy()

            for field, weight in self.field_weights.items():
                if weight <= 0:
                    continue
                doc_col = f'sparse_{field}'
                if doc_col not in docs_df.columns:
                    continue
                tokenized_corpus = [text.split() for text in docs_df[doc_col].fillna('')]
                if not any(tokenized_corpus):
                    continue
                self.logger.info(f'BM25 field: {field} | weight={weight}')
                model = BM25Plus(tokenized_corpus, k1=self.k1, b=self.b, delta=self.delta)
                field_models.append((weight, model))

            results = []
            for row in queries_df.itertuples(index=False):
                query_tokens = getattr(row, query_col).split()
                scores = np.zeros(len(docs_df), dtype=np.float32)
                for weight, model in field_models:
                    scores += weight * model.get_scores(query_tokens)
                top_idx = top_k_indices_desc(scores, top_k)
                results.append(build_result(row.id, doc_ids[top_idx].tolist()))
            return results


class HybridEmbeddingRetriever:
    def __init__(self, tfidf: WeightedTfidfRetriever, bm25: WeightedBM25Retriever, runtime: RuntimeEnvironment, config: EmbeddingConfig, logger: StructuredLogger) -> None:
        self.tfidf = tfidf
        self.bm25 = bm25
        self.runtime = runtime
        self.config = config
        self.logger = logger

    def _load_model(self):
        from sentence_transformers import SentenceTransformer
        device = self.runtime.resolve_embedding_device()
        self.logger.info(f'Embedding device: {device}')
        self.logger.info(f'Embedding model: {self.config.model_name} | batch_size: {self.config.batch_size} | candidate_k: {self.config.candidate_k}')
        return SentenceTransformer(self.config.model_name, device=device)

    def search(self, docs_df: pd.DataFrame, queries_df: pd.DataFrame, top_k: int) -> List[dict]:
        with self.logger.section('Hybrid embedding retrieval'):
            candidate_k = min(max(self.config.candidate_k, top_k), len(docs_df))
            self.logger.info(f'Queries: {len(queries_df):,} | Docs: {len(docs_df):,} | top_k: {top_k} | candidate_k: {candidate_k}')

            tfidf_results = self.tfidf.search(docs_df, queries_df, top_k=candidate_k)
            bm25_results = self.bm25.search(docs_df, queries_df, top_k=candidate_k)

            model = self._load_model()
            with self.logger.section('Embedding encode: documents'):
                doc_embeddings = model.encode(
                    docs_df['content'].tolist(),
                    batch_size=self.config.batch_size,
                    show_progress_bar=True,
                    normalize_embeddings=True,
                )
            with self.logger.section('Embedding encode: queries'):
                query_embeddings = model.encode(
                    queries_df['content'].tolist(),
                    batch_size=self.config.batch_size,
                    show_progress_bar=True,
                    normalize_embeddings=True,
                )

            doc_ids = docs_df['id'].astype(str).to_numpy()
            doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}
            query_ids = queries_df['id'].astype(str).tolist()
            results = []
            progress_interval = max(1, len(query_embeddings) // 5)

            with self.logger.section('Hybrid reranking'):
                for query_idx, query_embedding in enumerate(query_embeddings):
                    candidate_doc_ids = []
                    seen = set()
                    for ranked_doc_ids in (tfidf_results[query_idx]['relevant_docs'], bm25_results[query_idx]['relevant_docs']):
                        for doc_id in ranked_doc_ids:
                            doc_id = str(doc_id)
                            if doc_id in seen:
                                continue
                            candidate_doc_ids.append(doc_id)
                            seen.add(doc_id)

                    candidate_indices = np.array([doc_id_to_index[doc_id] for doc_id in candidate_doc_ids], dtype=int)
                    candidate_scores = doc_embeddings[candidate_indices] @ query_embedding
                    reranked_idx = top_k_indices_desc(candidate_scores, top_k)
                    ranked_doc_ids = [candidate_doc_ids[idx] for idx in reranked_idx]
                    results.append(build_result(query_ids[query_idx], ranked_doc_ids))

                    if (query_idx + 1) % progress_interval == 0 or query_idx + 1 == len(query_embeddings):
                        self.logger.info(f'Hybrid reranking progress: {query_idx + 1}/{len(query_embeddings)} queries')

            return results


class SubmissionWriter:
    def __init__(self, logger: StructuredLogger) -> None:
        self.logger = logger

    def write(self, results: Sequence[dict], sample_csv_path: Path, output_csv_path: Path) -> None:
        with self.logger.section('Submission writing'):
            self.logger.info(f'Output path: {output_csv_path}')
            predictions = {str(item['query_id']): [str(doc_id) for doc_id in item['relevant_docs']] for item in results}

            with sample_csv_path.open('r', newline='', encoding='utf-8') as source_file:
                reader = csv.DictReader(source_file)
                fieldnames = reader.fieldnames
                rows = list(reader)

            if fieldnames is None or len(fieldnames) < 2:
                raise ValueError('Sample submission format is invalid.')

            query_id_col = fieldnames[0]
            prediction_col = fieldnames[1]
            category_col = fieldnames[2] if len(fieldnames) >= 3 else None

            output_csv_path.parent.mkdir(parents=True, exist_ok=True)
            with output_csv_path.open('w', newline='', encoding='utf-8') as target_file:
                writer = csv.DictWriter(target_file, fieldnames=fieldnames)
                writer.writeheader()
                for row in rows:
                    query_id = str(row[query_id_col])
                    if query_id not in predictions:
                        raise ValueError(f'Missing prediction for query_id: {query_id}')
                    output_row = {
                        query_id_col: query_id,
                        prediction_col: json.dumps(predictions[query_id]),
                    }
                    if category_col is not None:
                        output_row[category_col] = row.get(category_col, '?') or '?'
                    writer.writerow(output_row)

            self.logger.info(f'Finished writing submission with {len(rows):,} rows')


class KaggleSubmissionPipeline:
    def __init__(self, config: PipelineConfig, logger: StructuredLogger) -> None:
        self.config = config
        self.logger = logger
        self.runtime = RuntimeEnvironment(config, logger)
        self.loader = JsonDatasetLoader(logger)
        self.preprocessor = TextPreprocessor(config.sparse, logger)
        self.tfidf = WeightedTfidfRetriever(config.field_weights, logger)
        self.bm25 = WeightedBM25Retriever(config.field_weights, logger)
        self.hybrid = HybridEmbeddingRetriever(self.tfidf, self.bm25, self.runtime, config.embedding, logger)
        self.writer = SubmissionWriter(logger)

    def _prepare_frames(self, dataset: SubmissionDataset) -> SubmissionDataset:
        with self.logger.section('Frame preparation'):
            docs_df = self.preprocessor.create_content_column(dataset.docs_df, ['title', 'text', 'tags'])
            queries_df = self.preprocessor.create_content_column(dataset.queries_df, ['title', 'text'])

            docs_df = self.preprocessor.build_sparse_field_columns(docs_df, self.config.field_weights.keys())
            queries_df = self.preprocessor.build_sparse_field_columns(queries_df, ['title', 'text'])
            docs_df = self.preprocessor.build_sparse_content_column(docs_df)
            queries_df = self.preprocessor.build_sparse_content_column(queries_df)

            self.logger.info(f'docs_df shape: {docs_df.shape}')
            self.logger.info(f'queries_df shape: {queries_df.shape}')
            self.logger.info(f'Field weights: {self.config.field_weights}')
            self.logger.info(f'Sparse config: {self.config.sparse}')

            return SubmissionDataset(docs_df=docs_df, queries_df=queries_df, sample_submission_path=dataset.sample_submission_path)

    def _run_retrieval(self, docs_df: pd.DataFrame, queries_df: pd.DataFrame, model_name: str) -> List[dict]:
        top_k = min(self.config.top_k, len(docs_df))
        self.logger.info(f'Retrieval mode: {model_name} | top_k: {top_k}')
        if model_name == 'bm25':
            return self.bm25.search(docs_df, queries_df, top_k=top_k)
        if model_name == 'tfidf':
            return self.tfidf.search(docs_df, queries_df, top_k=top_k)
        if model_name == 'embedding_hybrid':
            return self.hybrid.search(docs_df, queries_df, top_k=top_k)
        raise ValueError('model_name must be one of: bm25, tfidf, embedding_hybrid, auto')

    def run(self) -> dict:
        started_at = time.perf_counter()
        data_dir = self.runtime.resolve_data_dir()
        output_path = self.runtime.resolve_output_path(data_dir)
        model_name = self.runtime.resolve_model_name()

        self.logger.info(f'Final output path: {output_path}')
        self.logger.info(f'Resolved model name: {model_name}')

        dataset = self.loader.load_submission_dataset(data_dir)
        prepared = self._prepare_frames(dataset)
        results = self._run_retrieval(prepared.docs_df, prepared.queries_df, model_name)
        self.writer.write(results, prepared.sample_submission_path, output_path)

        total_runtime = time.perf_counter() - started_at
        self.logger.info(f'Generated retrieval results for {len(results):,} queries')
        self.logger.info(f'Total pipeline runtime: {total_runtime:.2f}s')

        return {
            'data_dir': str(data_dir),
            'output_path': str(output_path),
            'model_name': model_name,
            'query_count': len(prepared.queries_df),
            'doc_count': len(prepared.docs_df),
            'runtime_seconds': round(total_runtime, 2),
        }


In [ ]:
LOGGER = StructuredLogger()

CONFIG = PipelineConfig(
    model_name='embedding_hybrid',
    top_k=100,
    output_path=Path('notebooks/kaggle/solutions_SeaFour.csv'),
    field_weights={'title': 3.0, 'text': 1.0, 'tags': 2.0},
    sparse=SparseConfig(
        lowercase=True,
        strip_noise=True,
        collapse_whitespace=True,
        remove_stopwords=False,
        normalizer='stem',
        normalize_numbers=True,
        normalize_units=True,
        normalize_hyphens=True,
    ),
    embedding=EmbeddingConfig(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        batch_size=128,
        candidate_k=200,
        device='auto',
    ),
)

PIPELINE = KaggleSubmissionPipeline(CONFIG, LOGGER)
RUN_SUMMARY = PIPELINE.run()
RUN_SUMMARY


In [ ]:
submission_preview = pd.read_csv(RUN_SUMMARY['output_path'])
submission_preview.head()
